# DFA comparisons: SDS and language similarity

Edit the configuration cell, inspect the preview and benchmark, then run the
full-experiment cell. All calculations and file handling live in sibling modules.
Requires Python 3.12 and a Jupyter Python kernel; calculation modules use only
the standard library. Start Jupyter from the repository root or `code/`.

SDS is the shortest word accepted by exactly one DFA. Similarity samples length
uniformly from 0 through H, then samples symbols independently and uniformly.
Agreement includes jointly rejected words; weighted Jaccard compares accepted
words. A score of 1 through H does not establish language equivalence.

Existing canonical SDS results have already been examined. Similarities on
these data are descriptive. Pairs sharing a DFA are dependent observations.
See `../similarity-metrics.md` for the definitions and validation requirements.


In [36]:
RUN_ID = "2026-11-15_21-32-55-118406_UTC"
ALPHABET_SIZES = [2, 3]  # None selects all alphabet sizes.
STATE_COUNTS = [5, 10, 15]  # None selects all state counts.
CALCULATIONS = "both"  # "sds", "similarity", or "both".
HORIZON = 5  # REQUIRED for similarity/both; set an integer >= 0. No research default chosen.
RESULTS_ID = None  # None creates a fresh UTC timestamp shared by the selected batches.
COMPRESS_OUTPUT = True
BENCHMARK_PAIRS = 100  # Uniformly sampled pairs PER batch; no result files written.
BENCHMARK_SEED = 20
DATA_ROOT_OVERRIDE = None  # Optional Path/string for another data directory.


In [37]:
import json
import logging
import sys
from pathlib import Path

search_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if (search_root / "experiments.py").is_file():
    CODE_DIR = search_root
elif (search_root / "code" / "experiments.py").is_file():
    CODE_DIR = search_root / "code"
else:
    raise RuntimeError("Start this notebook from the repository root or code/ directory")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import experiments
from result_summary import markdown_summary, summarize

DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE is not None else CODE_DIR.parent / "data"
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")


## Preview the selected data

Each batch is one alphabet size and one raw state count. Only unordered pairs
within that batch are compared. Self-pairs and reversed duplicates are excluded.
The older `2026-11-15_00-30-21-627385_UTC` run duplicates the canonical samples;
do not combine the two as independent data. No new DFAs are generated here.


In [38]:
selected_batches = experiments.preview(RUN_ID, DATA_ROOT, ALPHABET_SIZES, STATE_COUNTS)
for batch in selected_batches:
    print(f"Alphabet {batch['alphabet_size']}, states {batch['state_count']}: "
          f"{batch['dfa_count']:,} DFAs, {batch['pair_count']:,} pairs")
print(f"Total: {sum(b['pair_count'] for b in selected_batches):,} pairs")
print(f"Calculations: {CALCULATIONS}; horizon: {HORIZON}")


Alphabet 2, states 5: 500 DFAs, 124,750 pairs
Alphabet 2, states 10: 500 DFAs, 124,750 pairs
Alphabet 2, states 15: 500 DFAs, 124,750 pairs
Alphabet 3, states 5: 500 DFAs, 124,750 pairs
Alphabet 3, states 10: 500 DFAs, 124,750 pairs
Alphabet 3, states 15: 500 DFAs, 124,750 pairs
Total: 748,500 pairs
Calculations: both; horizon: 5


## Benchmark before running all pairs

Set HORIZON above before this cell if similarity is enabled. The benchmark uses
a reproducible uniform sample from each selected batch, and writes no files.
Estimates cover pair calculations only; statistics, gzip and disk I/O add time.
Runtime can grow substantially with H, alphabet size and reachable product size.


In [39]:
timings = experiments.benchmark(
    RUN_ID, DATA_ROOT, calculations=CALCULATIONS, horizon=HORIZON,
    alphabet_sizes=ALPHABET_SIZES, state_counts=STATE_COUNTS,
    sample_pairs=BENCHMARK_PAIRS, seed=BENCHMARK_SEED)
for timing in timings:
    print(f"Alphabet {timing['alphabet_size']}, states {timing['state_count']}: "
          f"{timing['sample_pairs']} sampled pairs in {timing['sample_seconds']:.3f}s; "
          f"estimated full pair calculation {timing['estimated_pair_seconds']:.1f}s")
print(f"Total estimated pair calculation: {sum(t['estimated_pair_seconds'] for t in timings):.1f}s")


Alphabet 2, states 5: 100 sampled pairs in 0.004s; estimated full pair calculation 5.0s
Alphabet 2, states 10: 100 sampled pairs in 0.005s; estimated full pair calculation 6.2s
Alphabet 2, states 15: 100 sampled pairs in 0.004s; estimated full pair calculation 5.4s
Alphabet 3, states 5: 100 sampled pairs in 0.006s; estimated full pair calculation 7.3s
Alphabet 3, states 10: 100 sampled pairs in 0.010s; estimated full pair calculation 11.9s
Alphabet 3, states 15: 100 sampled pairs in 0.013s; estimated full pair calculation 15.9s
Total estimated pair calculation: 51.7s


## Run the full experiment

This cell writes new results under each source batch's
`comparisons/<results_id>/`. Existing results are never overwritten. A failed
batch removes its temporary output; earlier completed batches remain valid.
To retry a failed multi-batch experiment, use a fresh results ID and inspect only
that ID. Existing `pairs/` canonical SDS results are preserved.

Use Restart Kernel and Run All for a reproducible final run after setting H.
Re-executing the run cell with RESULTS_ID=None creates another result set.


In [40]:
result_dirs = experiments.run_all(
    RUN_ID, DATA_ROOT, calculations=CALCULATIONS, horizon=HORIZON,
    results_id=RESULTS_ID, alphabet_sizes=ALPHABET_SIZES,
    state_counts=STATE_COUNTS, compress=COMPRESS_OUTPUT)
print("Results ID:", result_dirs[0].name)
for result_dir in result_dirs:
    print(result_dir)


2026-11-15 15:33:59,623 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC: 1000 pairs completed
2026-11-15 15:33:59,699 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC: 2000 pairs completed
2026-11-15 15:33:59,763 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC: 3000 pairs completed
2026-11-15 15:33:59,829 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC: 4000 pairs completed
2026-11-15 15:33:59,898 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC: 5000 pairs completed
2026-11-15 15:33:59,961 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC: 6000 pa

Results ID: 2026-11-15_21-33-59-533782_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_5\2026-11-15_21-32-55-118406_UTC\comparisons\2026-11-15_21-33-59-533782_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_10\2026-11-15_21-32-55-118406_UTC\comparisons\2026-11-15_21-33-59-533782_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\states_15\2026-11-15_21-32-55-118406_UTC\comparisons\2026-11-15_21-33-59-533782_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_3\states_5\2026-11-15_21-32-55-118406_UTC\comparisons\2026-11-15_21-33-59-533782_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_3\states_10\2026-11-15_21-32-55-118406_UTC\comparisons\2026-11-15_21-33-59-533782_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_3\states_15\2026-11-15_21-32-55-118406_UTC\comparisons\2026-11-15_21-33-59-5337

## Inspect saved results

Summaries read exactly the directories returned by this run. To inspect a prior
run without calculating again, skip the Run cell and set result_dirs explicitly:

    result_dirs = [DATA_ROOT / "alphabet_2" / "states_10" / RUN_ID /
                   "comparisons" / "YOUR_RESULTS_ID"]

For legacy SDS results, use `pairs` instead of `comparisons`. Summary helpers
also understand the existing canonical SDS format. Equivalent pairs remain
separate from length zero. Positive-length frequencies are conditional on L>0.


In [ ]:
summaries = [summarize(path) for path in result_dirs]
for summary in summaries:
    report = markdown_summary(summary)
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(report)
    else:
        display(Markdown(report))


## OPTIONAL: Compare different state sizes within each alphabet

Compare **10 vs 20**, **10 vs 30**, and **20 vs 30** states, separately for each
selected alphabet. States 15 and 25 are ignored, regardless of STATE_COUNTS.
Each combination uses every DFA from one batch against every DFA from the
other: 500 DFAs per batch gives 250,000 pairs per combination, 750,000 per
alphabet (3,000,000 for four alphabets). Matching numeric DFA IDs across batches
are distinct DFAs and are included. Reverse combinations are not repeated.

This section computes SDS, agreement, and weighted Jaccard together, using the
same bounded distribution as above. Set ENABLE_CROSS_STATE=True, choose the
source run, alphabets, and horizon below, then run the preview/benchmark cell.
Run the final cell when ready for the full computation. Leave the flag False
to skip this section during Run All. Setup is required; earlier within-batch
calculations do not need to be rerun. All three state batches must be present
for each selected alphabet. The defaults below use the newer four-alphabet
source run with H=5, matching its saved within-batch similarity results.

Results are stored separately under
`data/alphabet_<A>/cross_state/<source_run>/<results_id>/states_<S1>_vs_<S2>/`.
Each directory has streamed pair records, DFA stats, and a provenance manifest.
Existing results are never overwritten; after interruption, completed
combinations remain valid and retries need a fresh CROSS_RESULTS_ID.
Summaries keep equivalence separate from length zero. Comparisons sharing a
DFA remain dependent observations, and these analyses are descriptive.
The optional cells refresh their helper modules so an already-running kernel
picks up code changes without rerunning the earlier experiments.


In [33]:
ENABLE_CROSS_STATE = True
CROSS_RUN_ID = "2026-11-15_19-36-35-433280_UTC"
CROSS_ALPHABET_SIZES = None  # All alphabets in CROSS_RUN_ID; optionally select a subset.
CROSS_HORIZON = 5  # Matches saved within-batch metrics for this source run.
CROSS_RESULTS_ID = None  # Fresh timestamp; independent of within-batch RESULTS_ID.


In [34]:
if ENABLE_CROSS_STATE:
    import importlib
    import experiments
    import cross_state

    # Reload dependencies first: cross_state imports helpers from experiments.
    importlib.reload(experiments)
    importlib.reload(cross_state)

    cross_preview = cross_state.preview(CROSS_RUN_ID, DATA_ROOT, CROSS_ALPHABET_SIZES)
    for row in cross_preview:
        print(f"Alphabet {row['alphabet_size']}, states {row['state_count_a']} vs "
              f"{row['state_count_b']}: {row['dfa_count_a']:,} x {row['dfa_count_b']:,} "
              f"DFAs = {row['pair_count']:,} pairs")
    print(f"Total: {sum(row['pair_count'] for row in cross_preview):,} pairs; H={CROSS_HORIZON}")
    cross_timings = cross_state.benchmark(
        CROSS_RUN_ID, DATA_ROOT, horizon=CROSS_HORIZON,
        alphabet_sizes=CROSS_ALPHABET_SIZES, sample_pairs=BENCHMARK_PAIRS, seed=BENCHMARK_SEED)
    for row in cross_timings:
        print(f"Alphabet {row['alphabet_size']}, states {row['state_count_a']} vs "
              f"{row['state_count_b']}: {row['sample_pairs']} sampled pairs in "
              f"{row['sample_seconds']:.3f}s; estimated pair calculation "
              f"{row['estimated_pair_seconds'] / 60:.2f} min")
    print(f"Total estimate: {sum(row['estimated_pair_seconds'] for row in cross_timings) / 60:.2f} min "
          "(pair calculations only; stats, gzip, and disk I/O add time)")
else:
    print("Optional cross-state comparisons disabled. Set ENABLE_CROSS_STATE=True to enable.")


Alphabet 2, states 10 vs 20: 500 x 500 DFAs = 250,000 pairs
Alphabet 2, states 10 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 2, states 20 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 3, states 10 vs 20: 500 x 500 DFAs = 250,000 pairs
Alphabet 3, states 10 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 3, states 20 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 4, states 10 vs 20: 500 x 500 DFAs = 250,000 pairs
Alphabet 4, states 10 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 4, states 20 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 5, states 10 vs 20: 500 x 500 DFAs = 250,000 pairs
Alphabet 5, states 10 vs 30: 500 x 500 DFAs = 250,000 pairs
Alphabet 5, states 20 vs 30: 500 x 500 DFAs = 250,000 pairs
Total: 3,000,000 pairs; H=5
Alphabet 2, states 10 vs 20: 100 sampled pairs in 0.005s; estimated pair calculation 0.21 min
Alphabet 2, states 10 vs 30: 100 sampled pairs in 0.007s; estimated pair calculation 0.28 min
Alphabet 2, states 20 vs 30: 100 sampled pairs in 0.005s; estima

In [35]:
if ENABLE_CROSS_STATE:
    import importlib
    import experiments
    import cross_state
    import result_summary

    importlib.reload(experiments)
    importlib.reload(cross_state)
    importlib.reload(result_summary)

    cross_result_dirs = cross_state.run_all(
        CROSS_RUN_ID, DATA_ROOT, horizon=CROSS_HORIZON, alphabet_sizes=CROSS_ALPHABET_SIZES,
        results_id=CROSS_RESULTS_ID, compress=COMPRESS_OUTPUT)
    print("Cross-state results ID:", cross_result_dirs[0].parent.name)
    for path in cross_result_dirs:
        print(path)
        report = result_summary.markdown_summary(result_summary.summarize(path))
        try:
            from IPython.display import Markdown, display
        except ImportError:
            print(report)
        else:
            display(Markdown(report))


2026-11-15 15:03:45,473 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_20: 1000 pairs completed
2026-11-15 15:03:45,556 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_20: 2000 pairs completed
2026-11-15 15:03:45,622 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_20: 3000 pairs completed
2026-11-15 15:03:45,694 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_20: 4000 pairs completed
2026-11-15 15:03:45,767 INFO: c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-

Cross-state results ID: 2026-11-15_21-03-45-337283_UTC
c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_20


### Alphabet 2, states 10 vs 20

Results: `2026-11-15_21-03-45-337283_UTC`; pairs: 250,000.

Similarity horizon: 5 (uniform length and symbols).

Equivalent pairs: 0 (0.000%).

| SDS length | Count | Fraction of finite lengths | Fraction conditional on L>0 |
|---|---:|---:|---:|
| 0 | 125,024 | 50.010% | — |
| 1 | 93,427 | 37.371% | 74.756% |
| 2 | 29,341 | 11.736% | 23.477% |
| 3 | 2,144 | 0.858% | 1.716% |
| 4 | 60 | 0.024% | 0.048% |
| 5 | 4 | 0.002% | 0.003% |

| SDS group | Pairs | Mean agreement | Mean weighted Jaccard |
|---|---:|---:|---:|
| All | 250,000 | 0.500298 | 0.307628 |
| 0 | 125,024 | 0.414885 | 0.241221 |
| 1 | 93,427 | 0.555815 | 0.347618 |
| 2 | 29,341 | 0.666455 | 0.445783 |
| 3 | 2,144 | 0.775906 | 0.542701 |
| 4 | 60 | 0.897396 | 0.408250 |
| 5 | 4 | 0.981771 | 0.981771 |

| DFA statistic | Size: count |
|---|---|
| Reachable | 2: 1, 3: 1, 4: 3, 5: 22, 6: 49, 7: 86, 8: 131, 9: 146, 10: 88, 11: 20, 12: 21, 13: 41, 14: 69, 15: 75, 16: 100, 17: 75, 18: 42, 19: 23, 20: 7 |
| Minimal | 1: 11, 2: 3, 3: 2, 4: 13, 5: 28, 6: 55, 7: 90, 8: 127, 9: 125, 10: 78, 11: 19, 12: 31, 13: 47, 14: 71, 15: 79, 16: 89, 17: 68, 18: 38, 19: 20, 20: 6 |

c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_30


### Alphabet 2, states 10 vs 30

Results: `2026-11-15_21-03-45-337283_UTC`; pairs: 250,000.

Similarity horizon: 5 (uniform length and symbols).

Equivalent pairs: 0 (0.000%).

| SDS length | Count | Fraction of finite lengths | Fraction conditional on L>0 |
|---|---:|---:|---:|
| 0 | 125,032 | 50.013% | — |
| 1 | 93,468 | 37.387% | 74.794% |
| 2 | 29,288 | 11.715% | 23.436% |
| 3 | 2,192 | 0.877% | 1.754% |
| 4 | 20 | 0.008% | 0.016% |

| SDS group | Pairs | Mean agreement | Mean weighted Jaccard |
|---|---:|---:|---:|
| All | 250,000 | 0.500101 | 0.308784 |
| 2 | 29,288 | 0.664827 | 0.449921 |
| 0 | 125,032 | 0.416270 | 0.243420 |
| 1 | 93,468 | 0.554261 | 0.346534 |
| 3 | 2,192 | 0.768068 | 0.537960 |
| 4 | 20 | 0.868229 | 0.713951 |

| DFA statistic | Size: count |
|---|---|
| Reachable | 2: 1, 3: 1, 4: 2, 5: 22, 6: 48, 7: 85, 8: 128, 9: 142, 10: 72, 11: 1, 13: 1, 14: 3, 15: 3, 16: 6, 17: 13, 18: 17, 19: 34, 20: 57, 21: 66, 22: 55, 23: 71, 24: 58, 25: 49, 26: 33, 27: 21, 28: 10, 29: 1 |
| Minimal | 1: 11, 2: 3, 3: 2, 4: 12, 5: 28, 6: 54, 7: 89, 8: 122, 9: 122, 10: 58, 11: 1, 13: 1, 14: 5, 15: 4, 16: 9, 17: 12, 18: 20, 19: 37, 20: 63, 21: 61, 22: 57, 23: 69, 24: 50, 25: 51, 26: 31, 27: 20, 28: 7, 29: 1 |

c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_2\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_20_vs_30


... (15 outputs left out)

c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_5\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_10_vs_30


### Alphabet 5, states 10 vs 30

Results: `2026-11-15_21-03-45-337283_UTC`; pairs: 250,000.

Similarity horizon: 5 (uniform length and symbols).

Equivalent pairs: 0 (0.000%).

| SDS length | Count | Fraction of finite lengths | Fraction conditional on L>0 |
|---|---:|---:|---:|
| 0 | 125,036 | 50.014% | — |
| 1 | 121,038 | 48.415% | 96.858% |
| 2 | 3,926 | 1.570% | 3.142% |

| SDS group | Pairs | Mean agreement | Mean weighted Jaccard |
|---|---:|---:|---:|
| All | 250,000 | 0.499837 | 0.325068 |
| 1 | 121,038 | 0.581426 | 0.389217 |
| 0 | 125,036 | 0.415367 | 0.258222 |
| 2 | 3,926 | 0.674671 | 0.476287 |

| DFA statistic | Size: count |
|---|---|
| Reachable | 8: 1, 9: 36, 10: 463, 27: 2, 28: 10, 29: 93, 30: 395 |
| Minimal | 8: 1, 9: 37, 10: 462, 27: 2, 28: 10, 29: 93, 30: 395 |

c:\Users\hmark\CS_Projects\shortest-distinguishing-substring\data\alphabet_5\cross_state\2026-11-15_19-36-35-433280_UTC\2026-11-15_21-03-45-337283_UTC\states_20_vs_30


### Alphabet 5, states 20 vs 30

Results: `2026-11-15_21-03-45-337283_UTC`; pairs: 250,000.

Similarity horizon: 5 (uniform length and symbols).

Equivalent pairs: 0 (0.000%).

| SDS length | Count | Fraction of finite lengths | Fraction conditional on L>0 |
|---|---:|---:|---:|
| 0 | 125,024 | 50.010% | — |
| 1 | 121,058 | 48.423% | 96.865% |
| 2 | 3,918 | 1.567% | 3.135% |

| SDS group | Pairs | Mean agreement | Mean weighted Jaccard |
|---|---:|---:|---:|
| All | 250,000 | 0.500111 | 0.322409 |
| 1 | 121,058 | 0.581228 | 0.387921 |
| 0 | 125,024 | 0.416213 | 0.254180 |
| 2 | 3,918 | 0.671001 | 0.475451 |

| DFA statistic | Size: count |
|---|---|
| Reachable | 17: 1, 18: 12, 19: 70, 20: 417, 27: 2, 28: 10, 29: 93, 30: 395 |
| Minimal | 17: 1, 18: 12, 19: 70, 20: 417, 27: 2, 28: 10, 29: 93, 30: 395 |